In [13]:
import pandas as pd
import glob
import os
import numpy as np
from datetime import datetime
import random
import string
from sqlalchemy import create_engine
import bcrypt

# 예시: 사용자명=root, 비밀번호=1234, DB 이름=test_db, 포트=3306
db_user = 'linkleTest'
db_password = '1111'
db_host = 'localhost'
db_port = '3333'
db_name = 'linkleTest2'

# SQLAlchemy 엔진 생성 (pymysql 사용)
engine = create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')


# 0. address 및 인구수 데이터 활용
total_people = pd.read_csv(
    '/Users/isanghyeon/Downloads/행정구역_시군구_별__성별_인구수_20250828112413.csv',
    encoding='cp949'
)
total_people = total_people[['행정구역(시군구)별','2025.05']]
total_people


local_df = pd.read_csv('/Users/isanghyeon/Desktop/mentoTeam_AI/국토교통부_전국 법정동_20250415.csv')
local_df = local_df[['시도명','시군구명','읍면동명','리명','삭제일자']]

local_df['삭제일자'] = local_df['삭제일자'].replace(["NaN", "nan", "None", ""], np.nan)
local_df = local_df[local_df['삭제일자'].isna()]
local_df = local_df[['시도명','시군구명','읍면동명','리명']]
local_df = local_df[~local_df['시도명'].isin(['동해출장소', '북부출장소'])]
local_df = local_df[~local_df['시군구명'].isin([
    '동구안심출장','북구계양출장','북구칠곡출장','삼천포남양출',
    '서구검단출장','안양시동안출','안양시만안출','익산시함열출',
    '전주시효자출','중구영종출장','중구용유출장','증평출장소',
    '진해시옹동출','창원시마산회원구','청주시동부출','청주시서부출',
    '진행시웅천출','진행시웅동출','사천남양출장'
])]
local_df = local_df.dropna()


# 1. CSV 파일 병합
folder_path = '/Users/isanghyeon/Downloads/무제 폴더/'
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

df_list = [pd.read_csv(file) for file in csv_files]
category_df1 = pd.concat(df_list, ignore_index=True)

# 2. 필요한 열만 추출 + 중복 제거
category_df1 = category_df1[['RESPOND_ID','SEXDSTN_FLAG_CD','AGRDE_FLAG_NM','ANSWRR_OC_AREA_NM'
                             ,'INTRST_LSR_ACT_RN1_VALUE','INTRST_LSR_ACT_RN2_VALUE','INTRST_LSR_ACT_RN3_VALUE'
                             ,'INTRST_LSR_ACT_RN4_VALUE','INTRST_LSR_ACT_RN5_VALUE']]
df_subset_duplicates = category_df1.drop_duplicates(subset=['RESPOND_ID'])

# 3. 활동 리스트 추출
activity_list = df_subset_duplicates['INTRST_LSR_ACT_RN1_VALUE'].unique().tolist()

# 4. 카테고리 키워드 정의
# 기타 사항은 현재 개발 프로젝트에서 사용을 하지 않기 때문에 추후 제거
category_keywords = {
    '식사': ['요리', '맛집'],
    '카페': ['카페'],
    '음악': ['음악', '라디오'],
    '영화': ['영화', '영상 컨텐츠 시청'],
    '독서': ['독서', '책', '문학', '신문', '잡지'],
    '운동': ['수영', '헬스', '조깅', '걷기', '등산', '요가', '라켓', '구기', '스포츠', '격투', '골프',
           '싸이클', '댄스', '줄넘기', '스트레칭', '레저', '보디빌딩'],
    '음주': ['음주', '유흥'],
    '학습': ['자격증', '어학', '기술', '수강', '자기계발'],
    '쇼핑': ['쇼핑'],
    '병원': ['찜질방', '사우나', '미용'],
    '게임': ['게임', 'e스포츠', '보드게임', '갬블'],
    '여행': ['여행', '드라이브', '캠핑', '소풍', '관광', '테마파크', '축제']
}

# 5. 카테고리 매핑 함수 정의
def map_category(activity):
    for category, keywords in category_keywords.items():
        if any(keyword in str(activity) for keyword in keywords):
            return category
    return '기타'

# 6. 5개 항목에 대해 카테고리 리스트 생성
def map_all_categories(row):
    activities = [
        row['INTRST_LSR_ACT_RN1_VALUE'],
        row['INTRST_LSR_ACT_RN2_VALUE'],
        row['INTRST_LSR_ACT_RN3_VALUE'],
        row['INTRST_LSR_ACT_RN4_VALUE'],
        row['INTRST_LSR_ACT_RN5_VALUE']
    ]
    categories = [
        map_category(act) for act in activities if pd.notna(act)
    ]
    # '기타' 제거 + 중복 제거 + 순서 유지
    filtered = [cat for i, cat in enumerate(categories) if cat != '기타' and cat not in categories[:i]]
    return filtered
    
# 7. 적용
df_subset_duplicates['카테고리'] = df_subset_duplicates.apply(map_all_categories, axis=1)
numbers_to_add = np.arange(1, df_subset_duplicates.shape[0] + 1)
df_subset_duplicates['RESPOND_ID'] = numbers_to_add

# 8. 확인
display(df_subset_duplicates[['RESPOND_ID', '카테고리']])

/var/folders/ll/5jxc7b_575nbgskskr3r0qbr0000gn/T/ipykernel_91602/3039098474.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset_duplicates['카테고리'] = df_subset_duplicates.apply(map_all_categories, axis=1)
/var/folders/ll/5jxc7b_575nbgskskr3r0qbr0000gn/T/ipykernel_91602/3039098474.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset_duplicates['RESPOND_ID'] = numbers_to_add


,RESPOND_ID,카테고리
0,1,"[독서, 영화, 운동]"
1,2,[운동]
2,3,"[게임, 독서, 운동]"
3,4,"[영화, 운동]"
4,5,"[영화, 게임, 운동]"
...,...,...
55155,20318,[독서]
55180,20319,"[독서, 영화]"
55203,20320,[]
55205,20321,"[영화, 쇼핑, 식사, 독서, 여행]"


In [16]:
from __future__ import annotations
import os, time, math, sqlite3, random, json, string
from datetime import datetime, timedelta  # ✅ timedelta 추가 (랜덤 생성일/참여일 계산)
from typing import Iterable

import numpy as np
import pandas as pd
import requests


# 0) 공통/지오코딩 유틸
KAKAO_KEY = "1a86971404ba50dc891f6a1c81c4d58e"
CACHE_DB = "geocode_cache.sqlite"

def _ensure_cache():
    with sqlite3.connect(CACHE_DB) as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS geocode_cache (
            address TEXT PRIMARY KEY,
            lat REAL,
            lng REAL,
            raw_json TEXT,
            updated_at TEXT
        )
        """)
        conn.commit()

def cache_get(address: str):
    with sqlite3.connect(CACHE_DB) as conn:
        cur = conn.execute("SELECT lat,lng,raw_json FROM geocode_cache WHERE address = ?", (address,))
        row = cur.fetchone()
        if row:
            lat, lng, raw = row
            return {"lat": lat, "lng": lng, "raw": json.loads(raw) if raw else None}
    return None

def cache_put(address: str, lat: float, lng: float, raw_json: dict | None):
    with sqlite3.connect(CACHE_DB) as conn:
        conn.execute(
            "REPLACE INTO geocode_cache(address,lat,lng,raw_json,updated_at) VALUES(?,?,?,?,?)",
            (address, lat, lng, json.dumps(raw_json, ensure_ascii=False) if raw_json else None, datetime.utcnow().isoformat())
        )
        conn.commit()

def kakao_geocode(address: str, max_retry: int = 4, qps_delay: float = 0.05):
    if not address or not isinstance(address, str):
        return None
    _ensure_cache()
    cached = cache_get(address)
    if cached:
        return cached
    if not KAKAO_KEY:
        return None
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
    params = {"query": address}
    for i in range(max_retry):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                docs = data.get("documents", [])
                if docs:
                    doc = docs[0]
                    x = float(doc["x"])
                    y = float(doc["y"])
                    out = {"lat": y, "lng": x, "raw": doc}
                    cache_put(address, y, x, doc)
                    time.sleep(qps_delay)
                    return out
                time.sleep(qps_delay)
                break
            elif resp.status_code in (429, 503):
                time.sleep(qps_delay * (2 ** i))
            else:
                time.sleep(qps_delay)
        except requests.RequestException:
            time.sleep(qps_delay * (2 ** i))
    return None

def bulk_geocode_unique(addresses: Iterable[str]) -> dict[str, dict | None]:
    uniq = [a.strip() for a in set(a for a in addresses if isinstance(a, str) and a.strip())]
    res: dict[str, dict | None] = {}
    
    print(f"🌐 [지오코딩] 총 {len(uniq)}개의 주소 변환 시작")

    for idx, addr in enumerate(uniq, 1):
        res[addr] = kakao_geocode(addr)
        if idx % 50 == 0 or idx == len(uniq):
            print(f"   ➤ 진행 상황: {idx}/{len(uniq)} 완료")
    
    print(f"✅ [지오코딩] 전체 주소 변환 완료\n")
    return res


def jitter_latlng(lat: float, lng: float, radius_m: float = 150.0) -> tuple[float, float]:
    if radius_m <= 0:
        return lat, lng
    theta = random.random() * 2 * math.pi
    r = radius_m * math.sqrt(random.random())
    dlat = (r * math.cos(theta)) / 111_111.0
    dlng = (r * math.sin(theta)) / (111_111.0 * max(math.cos(math.radians(lat)), 1e-6))
    return lat + dlat, lng + dlng


# 1) USER 테이블 생성

# -> 나이대를 일반 숫자만 추출 '40대' -> '40'
def convert_age_to_int(age_str):
    if isinstance(age_str, str) and '대' in age_str:
        return int(age_str.replace('대', ''))
    return 0

# -> 이메일 설정
def generate_random_email():
    email_list = ['naver.com','daum.net','nate.com', 'gmail.com', 'kakao.com', 'hanmail.com']
    return ''.join(random.choices(string.ascii_lowercase, k=9)) + '@' + random.choice(email_list)

# 유저 테이블 생성
def create_user_table(df: pd.DataFrame) -> pd.DataFrame:
    u = df.copy()
    u['age'] = u['AGRDE_FLAG_NM'].apply(convert_age_to_int)
    u['gender'] = u['SEXDSTN_FLAG_CD'].map({'M': '남성', 'F': '여성'})
    u['user_id'] = u['RESPOND_ID']
    u['name'] = 'User' + u['RESPOND_ID'].astype(str)
    u['email'] = u['RESPOND_ID'].apply(lambda _: generate_random_email())
    
    def quick_hash(password: str) -> str:
        return hashlib.sha256(password.encode()).hexdigest()
    
    # 변경
    u['password'] = u['RESPOND_ID'].apply(lambda _: quick_hash("password123"))
    u['nickname'] = '닉네임' + u['RESPOND_ID'].astype(str)
    u['image'] = None
    u['background'] = None
    u['memo'] = None
    u['bank_id'] = None
    u['account_number'] = None
    u['balance'] = 0
    u['created_date'] = datetime.now().date()
    return u[['user_id','name','email','password','nickname','age','gender',
              'image','background','memo','bank_id','account_number','balance','created_date']].drop_duplicates('user_id')


# 2) 시도 인구 정규화
def normalize_population(total_people: pd.DataFrame) -> pd.DataFrame:
    # 링커 생성시 지역별 인구수 별로 만들기 위한 인구수 정규화
    pop = total_people.rename(columns={'행정구역(시군구)별':'시도명', '2025.05':'인구'}).copy()
    pop['시도명'] = pop['시도명'].astype(str).str.strip()
    # 전국 인구수 제외
    pop = pop[~pop['시도명'].eq('전국')]
    pop['인구'] = pd.to_numeric(pop['인구'], errors='coerce')
    pop = pop.dropna(subset=['인구'])
    pop['비율'] = pop['인구'] / pop['인구'].sum()
    return pop[['시도명','인구','비율']]

# 3) LINKER 생성

def random_datetime_within_days(days):
    now = datetime.now()
    random_days = random.randint(0, days)
    random_seconds = random.randint(0, 86400)
    return now - timedelta(days=random_days, seconds=random_seconds)

def get_category_distribution(df: pd.DataFrame) -> pd.Series:
    # 카테고리 리스트 현재 [] ~ [1,2,3,4,5]의 형태의 리스트로 존재 지를 나눠 작업 진행
    exploded = df[['RESPOND_ID', '카테고리']].explode('카테고리').dropna()
    # 카테고리 값들의 카운트를 구해 정규화 진행 -> 비율을 가지고 카테고리별 가중치 부여를 위해
    category_counts = exploded['카테고리'].value_counts()
    category_ratio = category_counts / category_counts.sum()
    return category_ratio

def create_linker_table_weighted_global_total(df_src: pd.DataFrame,
                                              local_df: pd.DataFrame,
                                              pop_df_indexed: pd.DataFrame,
                                              total_linker_count: int = 1500,
                                              seed: int = 42,
                                              jitter_m: float = 150.0,
                                              created_within_days: int = 30   # ✅ NEW: 최근 N일 이내 임의 생성일
                                              ) -> pd.DataFrame:
    rng = random.Random(seed)
    
    # 카테고리별 갯수 정리
    category_ratio = get_category_distribution(df_src)
    # 지역 이름명 복사
    loc = local_df.dropna(subset=['시도명','시군구명','읍면동명']).copy()
    for col in ['시도명','시군구명','읍면동명','리명']:
        loc[col] = loc[col].fillna('').astype(str).str.strip()

    rows = []
    linker_id = 1

    for cat, cat_ratio in category_ratio.items():
        # total_linker_count의 수를 지정해 링커가 무의미하게 많아지는 것을 방지
        # cat_ratio를 활용해 카테고리의 비중별 링커 생성 활용
        per_category_total = int(total_linker_count * cat_ratio)
        if per_category_total == 0:
            continue
        # 카테고리별 가중치 지역별 가중치를 활용해 링커 생성 준비
        alloc_float = (pop_df_indexed['비율'] * per_category_total)
        alloc_floor = np.floor(alloc_float).astype(int)
        # 가중치 부여중 위에서 내림을 진행하여 남은 값은 추가 부여
        remainder = per_category_total - int(alloc_floor.sum())
        order = (alloc_float - alloc_floor).sort_values(ascending=False)
        alloc = alloc_floor.copy()
        
        if remainder > 0:
            alloc.loc[order.index[:remainder]] += 1
        # alloc에 어디 지역에 링커를 몇개를 만들지 정해둠
        for sido, cnt in alloc.items():
            if cnt <= 0:
                continue
            # 읍면동 데이터까지 가져옴
            cand = loc[loc['시도명'] == sido]
            if cand.empty:
                continue
            # 샘플 지역 설정
            sampled = cand.sample(n=cnt, replace=True,
                                  random_state=rng.randrange(1, 10**9))
            # 읍면동 주소만 가져가 카카오 지오코딩 좌표 설정 -> 이를 통해 좌표를 좀더 디테일하게 분배하고 한지역, 한점에 좌표가 생기는 것을 방지
            emd_keys = [f"{a['시도명']} {a['시군구명']} {a['읍면동명']}".strip() for _, a in sampled.iterrows()]
            emd_map = bulk_geocode_unique(emd_keys)
            for _, a in sampled.iterrows():
                emd_addr = f"{a['시도명']} {a['시군구명']} {a['읍면동명']}".strip()
                full_addr = f"{emd_addr} {a['리명']}".strip()
                g = emd_map.get(emd_addr)
                if g:
                    lat, lng = g["lat"], g["lng"]
                    lat, lng = jitter_latlng(lat, lng, radius_m=jitter_m)
                else:
                    lat = round(rng.uniform(33.0, 38.6), 6)
                    lng = round(rng.uniform(126.0, 129.6), 6)

                # ✅ NEW: 생성일을 최근 N일 내 무작위로 부여
                created_dt = random_datetime_within_days(created_within_days)

                # 링커 테이블에 값들을 저장
                rows.append({
                    'linker_id': linker_id,
                    '카테고리': cat,
                    'name': f"{cat}_모임_{linker_id}",
                    'adresss_name': f"{cat}_상호_{linker_id}",
                    'address': full_addr,
                    'address_detail': a['시군구명'],
                    'location_x': round(lng, 6),
                    'location_y': round(lat, 6),
                    'state': 'DELETED',
                    'category_id': None,
                    'created_date': created_dt,   # ✅ 기존: datetime.now().date() -> 무작위 생성일
                    'memo': f"{cat} 관련 모임입니다."
                })
                linker_id += 1

    linker_table = pd.DataFrame(rows)

    # ✅ NEW: 생성일 기준 정렬 후 linker_id 재부여 (생성일 순으로 ID 정렬)
    if not linker_table.empty:
        linker_table = linker_table.sort_values(['created_date', 'linker_id']).reset_index(drop=True)
        linker_table['linker_id'] = np.arange(1, len(linker_table) + 1)

        # 기존 로직 유지: 카테고리 -> category_id 매핑
        cat2id = {c: i+1 for i, c in enumerate(linker_table['카테고리'].unique())}
        linker_table['category_id'] = linker_table['카테고리'].map(cat2id)

    return linker_table


# 4) PARTICIPATE
def create_participate_table_many(df_src: pd.DataFrame,
                                  linker_table: pd.DataFrame,
                                  min_participants: int = 5,
                                  max_participants: int = 40,
                                  lam: int = 12,
                                  seed: int = 42,
                                  participate_delay_max_days: int = 3  # ✅ NEW: 생성일 이후 0~3일 내 참여일 부여
                                  ) -> pd.DataFrame:
    # 재현 가능한 무작위 선택에 대한 시드 생성
    rng = np.random.default_rng(seed)
    # 사용자 데이터를 가져와서 이를 가공 5개의 카테고리를 분배 시작 및 이름 변경
    users = df_src[['RESPOND_ID', '카테고리']].explode('카테고리').dropna()
    users = users.rename(columns={'RESPOND_ID': 'user_id'})
    users_by_cat = users.groupby('카테고리')['user_id'].apply(list).to_dict()
    # 관계 생성
    # rows => 참여 관계(링커-유저)를 저장할 리스트
    # seen => 중복 방지용 쌍 저장
    rows, seen = [], set()
    # 각 링크마다 참여자를 선정
    # 포아송 분포를 활용한 참여자수 k 랜덤 분포
    # 현실적인 순자 분포
    for _, lk in linker_table.iterrows():
        cat = lk['카테고리']
        lid = int(lk['linker_id'])
        candidates = users_by_cat.get(cat, [])
        if not candidates:
            continue
        k = int(rng.poisson(lam))
        k = max(min_participants, min(k, max_participants, len(candidates)))
        chosen = rng.choice(candidates, size=k, replace=False)

        # ✅ NEW: 참여일 = 생성일 + [0..participate_delay_max_days] 중 임의
        created_dt = pd.to_datetime(lk['created_date']).to_pydatetime()
        delay_days = int(rng.integers(0, participate_delay_max_days + 1))
        participated_dt = created_dt + timedelta(days=delay_days)
        # 미래일 방지 (테스트/데모 안정성)
        now_dt = datetime.now()
        if participated_dt > now_dt:
            participated_dt = now_dt

        for uid in chosen:
            key = (int(uid), lid)
            if key in seen: 
                continue
            seen.add(key)
            rows.append({
                'linker_id': lid,
                'user_id': int(uid),
                'participated_date': participated_dt.date()  # 기존 스키마 유지: date 저장
            })
    part = pd.DataFrame(rows)
    if not part.empty:
        part = part.drop_duplicates(subset=['user_id','linker_id'])
    return part


# 5) 엔드투엔드 실행 함수
# 전체 함수 시작
def build_all(df_subset_duplicates: pd.DataFrame,
              local_df: pd.DataFrame,
              total_people: pd.DataFrame,
              total_linker_count: int = 1500,
              seed: int = 42,
              jitter_m: float = 150.0,
              created_within_days: int = 30,
              participate_delay_max_days: int = 3):

    print("🧍‍♂️ [1/5] 사용자(user) 테이블 생성 중...")
    user_table = create_user_table(df_subset_duplicates)
    print(f"✅ 사용자 테이블 생성 완료 ({len(user_table):,}명)\n")

    print("🌍 [2/5] 인구(population) 정규화 중...")
    pop_df = normalize_population(total_people)
    print(f"✅ 인구 정규화 완료 (시도 수: {len(pop_df):,})\n")

    print("📍 [3/5] 링커(linker) 테이블 생성 중...")
    linker_table = create_linker_table_weighted_global_total(
        df_src=df_subset_duplicates,
        local_df=local_df,
        pop_df_indexed=pop_df.set_index('시도명'),
        total_linker_count=total_linker_count,
        seed=seed,
        jitter_m=jitter_m,
        created_within_days=created_within_days
    )
    print(f"✅ 링커 테이블 생성 완료 ({len(linker_table):,}개)\n")

    print("🤝 [4/5] 참여(participate) 테이블 생성 중...")
    participate_table = create_participate_table_many(
        df_src=df_subset_duplicates,
        linker_table=linker_table,
        min_participants=5,
        max_participants=40,
        lam=12,
        seed=seed,
        participate_delay_max_days=participate_delay_max_days
    )
    print(f"✅ 참여 테이블 생성 완료 ({len(participate_table):,}개)\n")

    print("🔗 [5/5] 카테고리 ID 매핑 중...")
    category_to_id = {
        "식사": 1, "카페": 2, "음악": 3, "영화": 4, "독서": 5, "운동": 6,
        "음주": 7, "학습": 8, "쇼핑": 9, "병원": 10, "게임": 11, "여행": 12
    }
    linker_table['category_id'] = linker_table['카테고리'].map(category_to_id)
    print("✅ 전체 테이블 생성 완료!\n")

    return user_table, linker_table, participate_table, pop_df



In [17]:
# 데이터프레임 준비가 완료되었을 경우 실행
user_table, linker_table, participate_table, pop_df = build_all(
    df_subset_duplicates=df_subset_duplicates,
    local_df=local_df,
    total_people=total_people,
    total_linker_count=1500,  # 전체 링커 수를 고정 (원하는 수로 조절 가능)
    seed=42,
    jitter_m=150.0  # 주소 위치에 랜덤 오차 부여 반경 (m)
)

# 결과 출력
print(f"총 유저 수: {len(user_table):,}")
print(f"총 링커 수: {len(linker_table):,}")
print(f"총 참여 수: {len(participate_table):,}")

# 카테고리별 링커 수 확인 
print("\n카테고리별 링커 수:")
print(linker_table['카테고리'].value_counts())

# 샘플 출력
print("\n샘플 주소 + 위경도:")
print(linker_table[['카테고리', 'address', 'location_y', 'location_x']].head())

🧍‍♂️ [1/5] 사용자(user) 테이블 생성 중...
✅ 사용자 테이블 생성 완료 (20,322명)

🌍 [2/5] 인구(population) 정규화 중...
✅ 인구 정규화 완료 (시도 수: 17)

📍 [3/5] 링커(linker) 테이블 생성 중...
🌐 [지오코딩] 총 5개의 주소 변환 시작
   ➤ 진행 상황: 5/5 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 8개의 주소 변환 시작
   ➤ 진행 상황: 8/8 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 12개의 주소 변환 시작
   ➤ 진행 상황: 12/12 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 6개의 주소 변환 시작
   ➤ 진행 상황: 6/6 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 1개의 주소 변환 시작
   ➤ 진행 상황: 1/1 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 58개의 주소 변환 시작
   ➤ 진행 상황: 50/58 완료
   ➤ 진행 상황: 58/58 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 9개의 주소 변환 시작
   ➤ 진행 상황: 9/9 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 10개의 주소 변환 시작
   ➤ 진행 상황: 10/10 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 13개의 주소 변환 시작
   ➤ 진행 상황: 13/13 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 9개의 주소 변환 시작
   ➤ 진행 상황: 9/9 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 11개의 주소 변환 시작
   ➤ 진행 상황: 11/11 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 14개의 주소 변환 시작
   ➤ 진행 상황: 14/14 완료
✅ [지오코딩] 전체 주소 변환 완료

🌐 [지오코딩] 총 19개의 주소 변환 시작
   ➤ 진행 상

In [18]:
display(user_table)
display(linker_table)
display(participate_table)
display(pop_df)

,user_id,name,email,password,nickname,age,gender,image,background,memo,bank_id,account_number,balance,created_date
0,1,User1,ofopvghhq@hanmail.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임1,60,여성,None,None,None,None,None,0,2025-09-10
1,2,User2,zdwsigsye@daum.net,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임2,50,남성,None,None,None,None,None,0,2025-09-10
2,3,User3,gyuaktjal@kakao.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임3,30,남성,None,None,None,None,None,0,2025-09-10
3,4,User4,aluauaczp@kakao.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임4,50,남성,None,None,None,None,None,0,2025-09-10
4,5,User5,yzgbzbpwr@gmail.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임5,20,남성,None,None,None,None,None,0,2025-09-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55155,20318,User20318,slzkucemr@kakao.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임20318,40,남성,None,None,None,None,None,0,2025-09-10
55180,20319,User20319,jcrihpyiz@naver.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임20319,40,여성,None,None,None,None,None,0,2025-09-10
55203,20320,User20320,sdrkpvihf@kakao.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임20320,60,남성,None,None,None,None,None,0,2025-09-10
55205,20321,User20321,lbvikjpie@hanmail.com,ef92b778bafe771e89245b89ecbc08a44a4e166c066599...,닉네임20321,40,여성,None,None,None,None,None,0,2025-09-10


,linker_id,카테고리,name,adresss_name,address,address_detail,location_x,location_y,state,category_id,created_date,memo
0,1,영화,영화_모임_520,영화_상호_520,경기도 포천시 일동면 화대리,포천시,127.317728,37.962367,DELETED,4,2025-08-10 14:29:32.158344,영화 관련 모임입니다.
1,2,음악,음악_모임_1042,음악_상호_1042,경상남도 거제시 하청면 어온리,거제시,128.655391,34.955858,DELETED,3,2025-08-10 14:36:10.272236,음악 관련 모임입니다.
2,3,영화,영화_모임_499,영화_상호_499,경기도 광주시 남한산성면 하번천리,광주시,127.243473,37.464059,DELETED,4,2025-08-10 15:34:45.158069,영화 관련 모임입니다.
3,4,영화,영화_모임_488,영화_상호_488,경기도 화성시 장안면 덕다리,화성시,126.831981,37.079888,DELETED,4,2025-08-10 15:35:36.157924,영화 관련 모임입니다.
4,5,음주,음주_모임_1120,음주_상호_1120,부산광역시 기장군 정관읍 매학리,기장군,129.179302,35.325819,DELETED,7,2025-08-10 15:48:29.298835,음주 관련 모임입니다.
...,...,...,...,...,...,...,...,...,...,...,...,...
1132,1133,식사,식사_모임_820,식사_상호_820,전북특별자치도 임실군 운암면 선거리,임실군,127.159823,35.661644,DELETED,1,2025-09-10 11:15:45.215824,식사 관련 모임입니다.
1133,1134,병원,병원_모임_1115,병원_상호_1115,경상북도 성주군 성주읍 대황리,성주군,128.287750,35.919292,DELETED,10,2025-09-10 11:24:59.296805,병원 관련 모임입니다.
1134,1135,운동,운동_모임_142,운동_상호_142,경기도 평택시 청북읍 현곡리,평택시,126.934215,37.040718,DELETED,6,2025-09-10 13:28:22.093380,운동 관련 모임입니다.
1135,1136,쇼핑,쇼핑_모임_879,쇼핑_상호_879,경기도 여주시 대신면 천남리,여주시,127.587267,37.374357,DELETED,9,2025-09-10 13:34:26.231010,쇼핑 관련 모임입니다.


,linker_id,user_id,participated_date
0,1,17037,2025-08-12
1,1,10463,2025-08-12
2,1,8923,2025-08-12
3,1,17398,2025-08-12
4,1,15990,2025-08-12
...,...,...,...
13714,1137,9473,2025-09-10
13715,1137,3300,2025-09-10
13716,1137,10061,2025-09-10
13717,1137,3094,2025-09-10


,시도명,인구,비율
2,서울특별시,9328042.0,0.182298
3,부산광역시,3254457.0,0.063602
4,대구광역시,2357997.0,0.046082
5,인천광역시,3037049.0,0.059353
6,광주광역시,1399880.0,0.027358
7,대전광역시,1439607.0,0.028134
8,울산광역시,1094027.0,0.021381
9,세종특별자치시,392211.0,0.007665
10,경기도,13706488.0,0.267866
11,강원특별자치도,1511341.0,0.029536


In [19]:
# user_id 제거
if 'user_id' in user_table.columns:
    user_table = user_table.drop(columns=['user_id'])

# linker_id 제거
if 'linker_id' in linker_table.columns:
    linker_table = linker_table.drop(columns=['linker_id'])

# participate_id 제거 (있다면)
if 'participate_id' in participate_table.columns:
    participate_table = participate_table.drop(columns=['participate_id'])
    
linker_table = linker_table.rename(columns={'adresss_name': 'address_name'})

user_table.to_sql(name='user', con=engine, index=False, if_exists='append')
cols_for_db = [
    'name','address_name','address','address_detail',
    'location_x','location_y','state','category_id','created_date','memo'
]

linker_table[cols_for_db].to_sql(
    name='linker',
    con=engine,
    index=False,
    if_exists='append'
)
participate_table.to_sql(name='participate', con=engine, index=False, if_exists='append')

# # 인구 비율 테이블 (선택사항)
# pop_df.to_sql(name='population_table', con=engine, index=False, if_exists='replace')

13719

In [24]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import requests

# .env 파일 로드
# DB 정보를 .env 파일로 관리하여 보안 강화
load_dotenv()

# 환경변수로부터 값 읽기
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')


# SQLAlchemy 엔진 생성 (pymysql 사용)
engine = create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

# 학습 테이블 DB에서 불러오기 -> DataFrame 형태
user_table = pd.read_sql('SELECT * FROM USER', con=engine)
linker_table = pd.read_sql('SELECT * FROM LINKER', con=engine)
participate_table = pd.read_sql('SELECT * FROM PARTICIPATE', con=engine)

def get_top_address_detail(user_id: int, engine) -> str | None:
    query = """
    SELECT sub.address_detail
    FROM (
        SELECT 
            l.address_detail,
            COUNT(*) AS cnt,
            MAX(p.participated_date) AS latest_date
        FROM PARTICIPATEtest p
        INNER JOIN LINKLERtest l
            ON p.linker_id = l.linker_id
        WHERE p.user_id = %(user_id)s
        GROUP BY l.address_detail
    ) sub
    ORDER BY sub.cnt DESC, sub.latest_date DESC
    LIMIT 1;
    """
    df = pd.read_sql(query, con=engine, params={"user_id": user_id})
    if df.empty:
        return None
    return df.iloc[0]['address_detail']

# 유저-링커 행렬 생성
# crosstab을 활용하여 배열에 대한 단순 교차표를 만든다.
# 크로스탭(crosstab)은 두 개 이상의 요인(factors)에 대한 교차표를 간단하게 만드는 함수이다.
# 다양한 매개변수를 활용하여 다양한 가능성을 확인 할 수 있다.
user_item_matrix = pd.crosstab(participate_table['user_id'], participate_table['linker_id'])

# SVD(특이값 분해, Singular Value Decomposition) 모델 활용
# 행렬 분해 방법 중 하나로 매우 많은 feature를 가진 고차원 행렬을 저차원 행렬로 분리하는 기법이다.
# SVD(특이값 분해)를 사용하여 유저-링커 행렬을 저차원 잠재 요인(latent factor)으로 분해
svd = TruncatedSVD(n_components=10, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

# 예측 점수 행렬
predicted_ratings = np.dot(user_factors, item_factors.T)
predicted_df = pd.DataFrame(predicted_ratings,
                            index=user_item_matrix.index,
                            columns=user_item_matrix.columns)


# 중요도 설정시 나이대와 성별을 가지고 가중치 부여를 위한 작업
# 동일한 나이대와 성별을 가진 사용자들이 선호한 링커 정보를 기반으로
# 현재 사용자와 유사한 선호 경향에 대한 가중치 반환
def get_similar_user_preference(user_id, user_table, participate_table):
    target = user_table[user_table['user_id'] == user_id]
    if target.empty:
        return {}
    age = target.iloc[0]['age']
    gender = target.iloc[0]['gender']
    # 같은 성별+나이대 유저들
    sim_users = user_table[(user_table['age'] == age) & (user_table['gender'] == gender)]['user_id']
    sim_parts = participate_table[participate_table['user_id'].isin(sim_users)]
    return sim_parts['linker_id'].value_counts(normalize=True).to_dict()

# 전체적인 작업 진행
# SVD기반 점수 + 사용자 선호도 + 지역 필터링
def recommend_linkers_hybrid_local(user_id, address_detail, top_n=5):
    if user_id not in predicted_df.index:
        return f"{user_id}에 대한 추천 결과가 없습니다."

    # 1) SVD 점수
    scores = predicted_df.loc[user_id].copy()

    # 2) 이미 참여한 링커 제거
    already = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index.tolist()
    scores.drop(labels=already, inplace=True, errors='ignore')

    # 3) 성별/나이대 가중치 반영
    weights = get_similar_user_preference(user_id, user_table, participate_table)
    for lid in scores.index:
        scores.loc[lid] += weights.get(lid, 0)

    # 4) 지역 필터
    local_ids = linker_table[linker_table['address_detail'] == address_detail]['linker_id']
    scores = scores[scores.index.isin(local_ids)]
    if scores.empty:
        return f"{address_detail} 지역에서 추천할 링커가 없습니다."

    # 5) Top-N
    top_ids = scores.sort_values(ascending=False).head(top_n).index
    return linker_table[linker_table['linker_id'].isin(top_ids)]

In [25]:
# 샘플 유저 선택
user_id_sample = user_table['user_id'].iloc[2]
# user_address_detail = linker_table['address_detail'].sample(1).iloc[0]
user_address_detail = '화성시'
print(user_address_detail)

recommended = recommend_linkers_hybrid_local(user_id_sample, user_address_detail, top_n=5)

print(f"[유저 {user_id_sample} | 지역 '{user_address_detail}' 추천 결과]")
print(recommended)


화성시
[유저 3 | 지역 '화성시' 추천 결과]
      linker_id          address address_detail address_name  category_id  \
9            10  경기도 화성시 정남면 문학리            화성시   학습_상호_1064            8   
150         151  경기도 화성시 향남읍 갈천리            화성시    게임_상호_942           11   
208         209  경기도 화성시 향남읍 제암리            화성시    게임_상호_944           11   
365         366  경기도 화성시 정남면 덕절리            화성시   음악_상호_1011            3   
1071       1072  경기도 화성시 양감면 용소리            화성시    독서_상호_300            5   

                   created_date  location_x  location_y          memo  \
9    2025-08-10 19:08:01.280027  126.971803   37.159956  학습 관련 모임입니다.   
150  2025-08-14 03:01:50.248701  126.919155   37.132541  게임 관련 모임입니다.   
208  2025-08-15 18:23:48.248727  126.921378   37.131837  게임 관련 모임입니다.   
365  2025-08-19 18:54:43.265072  126.970926   37.158910  음악 관련 모임입니다.   
1071 2025-09-08 18:11:10.127231  126.944796   37.080673  독서 관련 모임입니다.   

            name    state  
9     학습_모임_1064  DELETED  
150    게임_모임_9

Note: you may need to restart the kernel to use updated packages.
